In [65]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset , DataLoader
import pickle

In [66]:
data = datasets.MNIST('./Data',train = True , download = True)

In [67]:
X = data.data
y = data.targets

X_train , X_test , y_train , y_test = train_test_split(X,y,test_size = 0.2,shuffle = True)
X_train =  X_train.reshape(-1,28*28)

X_train = X_train.float()/255

Y_train = y_train.long()

x_test = X_test.float()
x_test = x_test.reshape(-1,28*28)/255

y_test = y_test.long()



In [72]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

device


device(type='cpu')

In [73]:
class Network(nn.Module):
  def __init__(self,size):
    super().__init__()
    self.linear = nn.Sequential(
        nn.Linear(size,100),
        nn.ReLU(),
        nn.Linear(100,75),
        nn.ReLU(),
        nn.Linear(75,10),
    )

  def forward(self,X):
    out = self.linear(X)
    return out

class Loader(Dataset):
  def __init__(self,x,y):
    self.x = x
    self.y = y
  def __len__(self):
    return self.x.size(0)
  def __getitem__(self,item):
    return self.x[item],self.y[item]


In [74]:
Train_loader = Loader(X_train,y_train)
Train_loader = DataLoader(Train_loader,batch_size = 32,shuffle = True)


In [75]:

model = Network(28*28).to(device)
optimizer = optim.Adam(model.parameters(),lr = 0.001)
criteria = nn.CrossEntropyLoss()

In [76]:
epoch = 20
for e in range(epoch):

  total_loss = 0

  for X,y in Train_loader:

    X = X.to(device)
    y = y.to(device)

    y_pred = model(X)

    loss = criteria(y_pred,y)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    total_loss += loss.item()
  print(f"epoch = {e+1} , loss = {total_loss/len(Train_loader)}")



epoch = 1 , loss = 0.32893915359179177
epoch = 2 , loss = 0.13929813659718882
epoch = 3 , loss = 0.09632272393993722
epoch = 4 , loss = 0.07154007375853447
epoch = 5 , loss = 0.05437669420890355
epoch = 6 , loss = 0.045921494912394945
epoch = 7 , loss = 0.03718006262922427
epoch = 8 , loss = 0.030683809009807494
epoch = 9 , loss = 0.026237779337493217
epoch = 10 , loss = 0.022460063117587803
epoch = 11 , loss = 0.019448852202436077
epoch = 12 , loss = 0.017131373299407338
epoch = 13 , loss = 0.016847032606165765
epoch = 14 , loss = 0.014808923774479277
epoch = 15 , loss = 0.013864954748306142
epoch = 16 , loss = 0.012264651442547651
epoch = 17 , loss = 0.012886926828207758
epoch = 18 , loss = 0.01073839622512825
epoch = 19 , loss = 0.012683107176234993
epoch = 20 , loss = 0.009674728068546093


In [77]:

with torch.no_grad():
  # 1. Get model predictions for all test items
  y_out = model(x_test)

  # 2. Get the index of the max value across the 10 outputs (dim=1)
  predictions = torch.argmax(y_out, dim=1)

  # 3. Count how many predictions match the true values
  count = (predictions == y_test).sum().item()

print("correct --> ", count, " accuracy -->", (count / len(y_test)) * 100, "%")


correct -->  11694  accuracy --> 97.45 %


In [78]:
print(torch.argmax(y_out,dim=1))
print(y_test)


tensor([1, 9, 4,  ..., 1, 7, 3])
tensor([1, 9, 4,  ..., 1, 7, 3])


In [79]:
pickle.dump(model,open('modelv2.pkl','wb'))